In [0]:
# ============================================================
# 1. SETUP E CREDENCIAIS (ecommerce_itens_pedido)
# ============================================================
import os
from dotenv import load_dotenv
from pyspark.sql.functions import current_timestamp, year, month, col, sum, round, date_format, when, regexp_replace, count, avg, ceil, lag
from pyspark.sql.window import Window
from pyspark.sql.window import Window

load_dotenv("../env")
client_id = os.getenv("CLIENT_ID")
tenant_id = os.getenv("TENANT_ID")
client_secret = os.getenv("CLIENT_SECRET")
storage_account = os.getenv("STORAGE_ACCOUNT_NAME")

adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

tabela = "ecommerce_itens_pedido"
path_raw = f"abfss://raw@{storage_account}.dfs.core.windows.net/batch-data/{tabela}.csv"
path_bronze = f"abfss://squad3@{storage_account}.dfs.core.windows.net/bronze/{tabela}"

print(f"Configuração finalizada para a tabela: {tabela}")

In [0]:
# ============================================================
# 2. EXTRAÇÃO E CARGA (RAW -> BRONZE)
# ============================================================
print(f"Lendo {tabela} da camada Raw...")

df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .options(**adls_options) 
    .load(path_raw)
)

timestamp_carga = current_timestamp()

df_bronze = (
    df_raw
    .withColumn("bronze_ingested_at", timestamp_carga)
    .withColumn("bronze_source_file", col("_metadata.file_path"))
    .withColumn("ano_particao", year(timestamp_carga))
    .withColumn("mes_particao", month(timestamp_carga))
)

print(f"Gravando fisicamente na camada Bronze: {path_bronze}")

(
    df_bronze.write
    .format("delta")
    .mode("append") 
    .options(**adls_options)
    .partitionBy("ano_particao", "mes_particao")
    .save(path_bronze)
)

print("Ingestão Bronze finalizada com sucesso!")

In [0]:
# ============================================================
# 3. CAMADA SILVER (ENRIQUECIMENTO E REGRAS COMPLEXAS)
# ============================================================

print(f"Iniciando processamento avançado da camada Silver para: {tabela}...")

# Caminhos das tabelas
path_silver_itens = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/{tabela}"
path_silver_produtos = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/ecommerce_produtos"
path_silver_pedidos = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/ecommerce_pedidos"

# 1. LEITURA (Bronze atual + Tabelas Silver auxiliares)
# 1. LEITURA (Bronze atual + Tabelas Silver auxiliares)
df_bronze_itens = spark.read.format("delta").options(**adls_options).load(path_bronze)
df_silver_produtos = spark.read.format("delta").options(**adls_options).load(path_silver_produtos)
df_silver_pedidos = spark.read.format("parquet").options(**adls_options).load(path_silver_pedidos)

# 2. LIMPEZA BÁSICA (Regra 1)
df_itens_clean = (
    df_bronze_itens
    .dropna(subset=["id_item_pedido"])
    .dropDuplicates(["id_item_pedido"])
    .withColumn("id_item_pedido", col("id_item_pedido").cast("string"))
    .withColumn("id_pedido", col("id_pedido").cast("string"))
    .withColumn("sku", col("sku").cast("string"))
)

# 3. JOINS PARA ENRIQUECIMENTO (Buscando dados necessários para as regras)
# Pegando apenas a unidade de medida dos produtos
df_prod_aux = df_silver_produtos.select("sku", col("unidade_medida").alias("aux_unidade_medida"))
df_itens_enriquecido = df_itens_clean.join(df_prod_aux, on="sku", how="left")

# Pegando a data do pedido (AJUSTE O NOME DA COLUNA 'dt_pedido' SE NECESSÁRIO)
df_ped_aux = df_silver_pedidos.select("id_pedido", col("dt_pedido").alias("aux_data_pedido"))
df_itens_enriquecido = df_itens_enriquecido.join(df_ped_aux, on="id_pedido", how="left")

# 4. APLICAÇÃO DAS REGRAS FINANCEIRAS E TIPAGEM (Regras 2, 3, 4 e 5)
df_silver = (
    df_itens_enriquecido
    
    # Regra 3: Tratamento e cast de preços para Decimal(10,2)
    .withColumn("preco_unitario", regexp_replace(col("preco_unitario"), ",", ".").cast("decimal(10,2)"))
    .withColumn("desconto_aplicado", regexp_replace(col("desconto_aplicado"), ",", ".").cast("decimal(10,2)"))
    
    # Regra 2: Quantidade condicional (INT para unidades/peças, FLOAT para kg/L)
    .withColumn("quantidade", 
        when(col("aux_unidade_medida").isin("un", "pc", "cx"), regexp_replace(col("quantidade"), ",", ".").cast("integer"))
        .otherwise(regexp_replace(col("quantidade"), ",", ".").cast("double"))
    )
    
    # Regra 4: Coluna derivada (Valor Líquido do Item)
    .withColumn("valor_liquido_item", 
        ((col("preco_unitario") - col("desconto_aplicado")) * col("quantidade")).cast("decimal(10,2)")
    )
    
    # Regra 5: Particionamento baseado na data do pedido (e não na data atual)
    .withColumn("ano_particao", year("aux_data_pedido"))
    .withColumn("mes_particao", month("aux_data_pedido"))
    
    # Auditoria
    .withColumn("silver_processed_at", current_timestamp())
)

# 5. SELEÇÃO FINAL (Removendo as colunas auxiliares que trouxemos dos Joins)
df_silver_final = df_silver.select(
    "id_item_pedido", "id_pedido", "sku", "quantidade", 
    "preco_unitario", "desconto_aplicado", "valor_liquido_item", 
    "ano_particao", "mes_particao", "silver_processed_at"
)


# 6. GRAVAÇÃO NA SILVER
print(f"Gravando dados refinados em: {path_silver_itens}")

(
    df_silver_final.write
    .format("delta")
    .mode("append")  
    .options(**adls_options)
    .partitionBy("ano_particao", "mes_particao")
    .save(path_silver_itens)
)

print("SUCESSO! Tabela de Itens salva na Silver com Joins, cálculos financeiros e particionamento histórico.\n")
display(df_silver_final.limit(5))

In [0]:
# ============================================================
# 4. CAMADA GOLD (DATA MARTS AGREGADOS - STAR SCHEMA)
# ============================================================

print("Iniciando construção dos Data Marts agregados na camada Gold...")

# ------------------------------------------------------------
# 1. LEITURA DAS TABELAS SILVER E JOINS BASE
# ------------------------------------------------------------
path_silver_produtos = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/ecommerce_produtos"
path_silver_categorias = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/ecommerce_categorias"

df_dim_produtos = spark.read.format("delta").options(**adls_options).load(path_silver_produtos)
df_dim_categorias = spark.read.format("delta").options(**adls_options).load(path_silver_categorias)

df_categorias_hierarquia = (
    df_dim_categorias.alias("sub")
    .join(
        df_dim_categorias.alias("raiz"),
        col("sub.id_categoria_pai") == col("raiz.id_categoria"),
        "left"
    )
    .select(
        col("sub.id_categoria").alias("id_categoria"),
        col("sub.nome_categoria").alias("nome_subcategoria"),
        col("raiz.nome_categoria").alias("nome_categoria_raiz")
    )
)

df_base = (
    df_silver_final.alias("f")
    .join(df_dim_produtos.alias("p"), on="sku", how="left")
    .join(df_categorias_hierarquia.alias("c"), on="id_categoria", how="left")
    .withColumn("trimestre", ceil(col("f.mes_particao") / 3))
    .withColumn("valor_bruto_item", (col("f.preco_unitario") * col("f.quantidade")).cast("decimal(10,2)"))
    .withColumn("valor_desconto_total", (col("f.desconto_aplicado") * col("f.quantidade")).cast("decimal(10,2)"))
)

# ------------------------------------------------------------
# 2. CÁLCULO DOS KPIs
# ------------------------------------------------------------
timestamp_formatado = date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")

df_kpi_categoria_mes = (
    df_base
    .groupBy("f.ano_particao", "f.mes_particao", "c.nome_categoria_raiz")
    .agg(
        round(sum("f.valor_liquido_item"), 2).alias("receita_liquida_total"),
        round(sum("valor_desconto_total") / sum("valor_bruto_item"), 4).alias("taxa_desconto_media")
    )
    .withColumn("gold_processed_at", timestamp_formatado)
)

df_kpi_sku_trimestre = (
    df_base
    .groupBy("f.ano_particao", "trimestre", "sku")
    .agg(
        round(sum("f.valor_liquido_item"), 2).alias("receita_total"),
        sum("f.quantidade").alias("quantidade_total")
    )
    .withColumn("gold_processed_at", timestamp_formatado)
)

df_kpi_complexidade_carrinho = (
    df_silver_final
    .groupBy("ano_particao", "mes_particao", "id_pedido")
    .agg(sum("quantidade").alias("total_itens_pedido"))
    .groupBy("ano_particao", "mes_particao")
    .agg(round(avg("total_itens_pedido"), 2).alias("media_itens_por_pedido"))
    .withColumn("gold_processed_at", timestamp_formatado)
)

window_yoy = Window.partitionBy("nome_subcategoria").orderBy("ano_particao")
df_kpi_yoy_subcategoria = (
    df_base
    .groupBy("f.ano_particao", "c.nome_subcategoria")
    .agg(round(sum("f.valor_liquido_item"), 2).alias("receita_ano_atual"))
    .withColumn("receita_ano_anterior", lag("receita_ano_atual").over(window_yoy))
    .withColumn("crescimento_yoy_perc", 
        round(((col("receita_ano_atual") - col("receita_ano_anterior")) / col("receita_ano_anterior")) * 100, 2)
    )
    .withColumn("gold_processed_at", timestamp_formatado)
)

# ------------------------------------------------------------
# 3. GRAVAÇÃO NO DELTA LAKE (PRODUÇÃO - APPEND)
# ------------------------------------------------------------
print("Gravando os Data Marts na camada Gold...")

data_marts = {
    "gold_kpi_receita_desconto_categoria": df_kpi_categoria_mes,
    "gold_kpi_sku_trimestre": df_kpi_sku_trimestre,
    "gold_kpi_complexidade_carrinho": df_kpi_complexidade_carrinho,
    "gold_kpi_yoy_subcategoria": df_kpi_yoy_subcategoria
}

for tabela_nome, df_mart in data_marts.items():
    path_gold = f"abfss://squad3@{storage_account}.dfs.core.windows.net/gold/{tabela_nome}"
    (
        df_mart.write
        .format("delta")
        .mode("append") 
        .option("mergeSchema", "true")
        .options(**adls_options)
        .save(path_gold)
    )
    print(f"Data Mart atualizado no Data Lake: {tabela_nome}")

print("\nSUCESSO! Data Marts sumarizados e salvos na Gold com sucesso.")

In [0]:
# ============================================================
# 5. EXPORTAÇÃO DE DATA MARTS PARA O SQL SERVER (PRODUÇÃO)
# ============================================================

print("Iniciando a exportação dos Data Marts para o SQL Server (MODO APPEND)...")

# 1. Carrega as credenciais do ambiente
load_dotenv("../env")
jdbc_hostname = os.getenv("SQL_HOST")
jdbc_port = "1433" 
jdbc_database = os.getenv("SQL_DATABASE")
jdbc_username = os.getenv("SQL_USERNAME")
jdbc_password = os.getenv("SQL_PASSWORD")

def carregar_no_sql_server(df, tabela_sql, mode="append"): # <--- Função com append nativo
    try:
        (
            df.write
            .format("sqlserver")
            .option("host", jdbc_hostname)
            .option("port", jdbc_port)
            .option("database", jdbc_database)
            .option("dbtable", tabela_sql)
            .option("user", jdbc_username)
            .option("password", jdbc_password)
            .mode(mode) 
            .save()
        )
        print(f"SUCESSO! Dados adicionados em {tabela_sql}")
    except Exception as e:
        print(f"Erro ao atualizar {tabela_sql}:\n{e}")

# 2. Mapeamento das tabelas
processamento = {
    "squad3.gold_kpi_receita_desconto_categoria": df_kpi_categoria_mes,
    "squad3.gold_kpi_sku_trimestre": df_kpi_sku_trimestre,
    "squad3.gold_kpi_complexidade_carrinho": df_kpi_complexidade_carrinho,
    "squad3.gold_kpi_yoy_subcategoria": df_kpi_yoy_subcategoria
}

# 3. Loop de carga
for tabela_destino, df_kpi in processamento.items():
    print(f"Iniciando carga: {tabela_destino}")
    carregar_no_sql_server(df_kpi, tabela_destino, mode="append") # <--- AQUI: Configurado explícito para append

print("\nTodas as tabelas foram atualizadas no schema squad3 do banco de dados!")